# Week 1 Day 5 — Final Validation, Error Analysis & Production Inference

## Project
Adult Income Classification

**Primary metric:** F1-score  
**Random state:** 42  
**Split:** 60% training, 20% development, 20% untouched final test  
**Goal:** Predict annual income `>50K` versus `<=50K`.


# Task 1: Data Preparation and Feature Definition

In this task, the dataset is prepared for the final machine-learning pipeline. Numerical and categorical features are defined separately, and the target is separated from the input features.


In [ ]:
import os
import sys
import random
import platform
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import sklearn

from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import learning_curve

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier

from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score
from sklearn.metrics import average_precision_score
from sklearn.metrics import brier_score_loss
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import roc_curve
from sklearn.metrics import precision_recall_curve

from sklearn.calibration import calibration_curve
from sklearn.datasets import fetch_openml

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

RESULTS = Path("results")
FIGURES = Path("figures")
MODELS = Path("models")

RESULTS.mkdir(exist_ok=True)
FIGURES.mkdir(exist_ok=True)
MODELS.mkdir(exist_ok=True)

COLUMNS = [
    "age",
    "workclass",
    "fnlwgt",
    "education",
    "education-num",
    "marital-status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "capital-gain",
    "capital-loss",
    "hours-per-week",
    "native-country",
    "income"
]

numeric_features = [
    "age",
    "fnlwgt",
    "education-num",
    "capital-gain",
    "capital-loss",
    "hours-per-week"
]

categorical_features = [
    "workclass",
    "education",
    "marital-status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "native-country"
]

def load_adult():
    candidates = [
        "adult.data",
        "adult.csv",
        "adult_income.csv",
        "adult_income_data.csv"
    ]

    for path in candidates:
        if not os.path.exists(path):
            continue

        if path.endswith(".data"):
            d = pd.read_csv(
                path,
                names=COLUMNS,
                skipinitialspace=True
            )
        else:
            d = pd.read_csv(path)

        return d, path

    d = fetch_openml(
        "adult",
        version=2,
        as_frame=True
    ).frame.copy()

    d.columns = [str(c).strip() for c in d.columns]

    return d, "OpenML"

df, data_source = load_adult()

df.columns = [str(c).strip() for c in df.columns]

if "class" in df.columns and "income" not in df.columns:
    df = df.rename(columns={"class": "income"})

df = df.replace("?", np.nan)

df["income"] = df["income"].astype(str).str.strip()

target_map = {
    "<=50K": 0,
    ">50K": 1,
    "<=50K.": 0,
    ">50K.": 1
}

df["target"] = df["income"].map(target_map)

df = df.dropna(subset=["target"]).copy()

X = df.drop(columns=["income", "target"])

y = df["target"].astype(int)

print("Dataset source:", data_source)
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Target distribution:")
print(y.value_counts())


## Conclusion

The Adult Income dataset has been prepared successfully. Numerical and categorical features are clearly separated, and the target variable is stored independently. This structure is ready for leakage-safe preprocessing, model training, validation, and final evaluation.


# Task 2: Leakage-Safe Train, Development and Test Split

The data is divided into training, development, and untouched final test sets. Stratification preserves the target distribution across the splits.


In [ ]:
X_train_dev, X_test, y_train_dev, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

X_train, X_dev, y_train, y_dev = train_test_split(
    X_train_dev,
    y_train_dev,
    test_size=0.25,
    stratify=y_train_dev,
    random_state=RANDOM_STATE
)

print("Training rows:", len(X_train))
print("Development rows:", len(X_dev))
print("Final test rows:", len(X_test))

print("Training target rate:", y_train.mean())
print("Development target rate:", y_dev.mean())
print("Test target rate:", y_test.mean())


## Conclusion

The final test set remains untouched during model development. The 60/20/20 split reduces evaluation leakage and provides a reliable final estimate after the model and decision threshold are locked.


# Task 3: Preprocessing Pipeline

Numerical features use median imputation and standardization. Categorical features use most-frequent imputation and one-hot encoding. Unknown categories are safely ignored.


In [ ]:
num_pipe = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

try:
    enc = OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    )
except TypeError:
    enc = OneHotEncoder(
        handle_unknown="ignore",
        sparse=False
    )

cat_pipe = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", enc)
    ]
)

preprocessor = ColumnTransformer(
    [
        ("numeric", num_pipe, numeric_features),
        ("categorical", cat_pipe, categorical_features)
    ]
)

print("Preprocessing pipeline created successfully.")


## Conclusion

All preprocessing steps are contained inside a reproducible pipeline. This prevents preprocessing information from the development or test sets from leaking into model training.


# Task 4: Model Training and Validation

Candidate models are trained using the same preprocessing pipeline. F1-score is used as the primary selection metric.


In [ ]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        random_state=RANDOM_STATE
    )
}

results = []

for name, model in models.items():
    pipe = Pipeline(
        [
            ("preprocessor", preprocessor),
            ("model", model)
        ]
    )

    pipe.fit(X_train, y_train)

    pred = pipe.predict(X_dev)

    if hasattr(pipe, "predict_proba"):
        prob = pipe.predict_proba(X_dev)[:, 1]
    else:
        prob = pipe.decision_function(X_dev)

    row = {
        "Model": name,
        "Accuracy": accuracy_score(y_dev, pred),
        "Precision": precision_score(y_dev, pred, zero_division=0),
        "Recall": recall_score(y_dev, pred, zero_division=0),
        "F1": f1_score(y_dev, pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_dev, prob),
        "Average_Precision": average_precision_score(y_dev, prob),
        "Brier": brier_score_loss(y_dev, prob)
    }

    results.append(row)

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    "F1",
    ascending=False
).reset_index(drop=True)

print(results_df)

results_df.to_csv(
    RESULTS / "model_comparison.csv",
    index=False
)


## Conclusion

The candidate models are compared using multiple metrics while keeping F1-score as the main selection criterion. The best development-set model can now be selected for final tuning and evaluation.


# Task 5: Final Model Selection

The model with the highest development F1-score is selected and fitted using the training data.


In [ ]:
best_name = results_df.loc[0, "Model"]

best_model = models[best_name]

final_pipeline = Pipeline(
    [
        ("preprocessor", preprocessor),
        ("model", best_model)
    ]
)

final_pipeline.fit(X_train, y_train)

dev_prob = final_pipeline.predict_proba(X_dev)[:, 1]

print("Selected model:", best_name)
print("Development F1:", f1_score(y_dev, (dev_prob >= 0.5).astype(int)))


## Conclusion

The highest-performing candidate on the development set has been selected. The final test set is still untouched.


# Task 6: Threshold Analysis

The default probability threshold of 0.50 is evaluated against alternative thresholds using the development set. The selected threshold is based only on development data.


In [ ]:
thresholds = np.arange(0.20, 0.81, 0.05)

threshold_rows = []

for threshold in thresholds:
    pred = (dev_prob >= threshold).astype(int)

    threshold_rows.append(
        {
            "Threshold": threshold,
            "Precision": precision_score(
                y_dev,
                pred,
                zero_division=0
            ),
            "Recall": recall_score(
                y_dev,
                pred,
                zero_division=0
            ),
            "F1": f1_score(
                y_dev,
                pred,
                zero_division=0
            )
        }
    )

threshold_df = pd.DataFrame(threshold_rows)

best_threshold = float(
    threshold_df.loc[
        threshold_df["F1"].idxmax(),
        "Threshold"
    ]
)

print(threshold_df)
print("Selected threshold:", best_threshold)

threshold_df.to_csv(
    RESULTS / "threshold_analysis.csv",
    index=False
)

plt.figure(figsize=(8, 5))
plt.plot(
    threshold_df["Threshold"],
    threshold_df["Precision"],
    marker="o",
    label="Precision"
)
plt.plot(
    threshold_df["Threshold"],
    threshold_df["Recall"],
    marker="o",
    label="Recall"
)
plt.plot(
    threshold_df["Threshold"],
    threshold_df["F1"],
    marker="o",
    label="F1"
)
plt.xlabel("Threshold")
plt.ylabel("Score")
plt.title("Threshold Analysis")
plt.legend()
plt.tight_layout()
plt.savefig(
    FIGURES / "threshold_analysis.png",
    dpi=150
)
plt.show()


## Conclusion

The decision threshold is selected using only the development set. This threshold is locked before evaluating the untouched final test set.


# Task 7: Final Test Evaluation

The locked pipeline and threshold are evaluated once on the untouched final test set.


In [ ]:
test_prob = final_pipeline.predict_proba(X_test)[:, 1]

test_pred = (
    test_prob >= best_threshold
).astype(int)

final_metrics = {
    "Accuracy": accuracy_score(y_test, test_pred),
    "Precision": precision_score(
        y_test,
        test_pred,
        zero_division=0
    ),
    "Recall": recall_score(
        y_test,
        test_pred,
        zero_division=0
    ),
    "F1": f1_score(
        y_test,
        test_pred,
        zero_division=0
    ),
    "ROC_AUC": roc_auc_score(y_test, test_prob),
    "Average_Precision": average_precision_score(
        y_test,
        test_prob
    ),
    "Brier": brier_score_loss(
        y_test,
        test_prob
    )
}

final_metrics_df = pd.DataFrame(
    [final_metrics]
)

print(final_metrics_df)

final_metrics_df.to_csv(
    RESULTS / "final_metrics.csv",
    index=False
)

print(classification_report(
    y_test,
    test_pred,
    target_names=["<=50K", ">50K"],
    zero_division=0
))


## Conclusion

The final test evaluation is performed only after model and threshold selection. These metrics provide the final unbiased performance estimate for the submitted pipeline.


# Task 8: Confusion Matrix and Error Analysis

False positives and false negatives are identified to understand the types of mistakes made by the final model.


In [ ]:
cm = confusion_matrix(
    y_test,
    test_pred
)

plt.figure(figsize=(6, 5))
plt.imshow(cm)
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.xticks([0, 1], ["<=50K", ">50K"])
plt.yticks([0, 1], ["<=50K", ">50K"])

for i in range(2):
    for j in range(2):
        plt.text(
            j,
            i,
            cm[i, j],
            ha="center",
            va="center"
        )

plt.tight_layout()
plt.savefig(
    FIGURES / "confusion_matrix.png",
    dpi=150
)
plt.show()

test_errors = X_test.copy()

test_errors["actual"] = y_test.values

test_errors["predicted"] = test_pred

test_errors["probability"] = test_prob

false_positive = test_errors[
    (test_errors["actual"] == 0)
    & (test_errors["predicted"] == 1)
]

false_negative = test_errors[
    (test_errors["actual"] == 1)
    & (test_errors["predicted"] == 0)
]

false_positive.to_csv(
    RESULTS / "false_positive_samples.csv",
    index=False
)

false_negative.to_csv(
    RESULTS / "false_negative_samples.csv",
    index=False
)

print("False positives:", len(false_positive))
print("False negatives:", len(false_negative))


## Conclusion

The confusion matrix and error samples show how the final classifier behaves on incorrect predictions. False positives and false negatives are saved for further inspection.


# Task 9: ROC and Precision-Recall Curves

ROC and Precision-Recall curves provide threshold-independent views of classifier performance.


In [ ]:
fpr, tpr, _ = roc_curve(
    y_test,
    test_prob
)

precision, recall, _ = precision_recall_curve(
    y_test,
    test_prob
)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr)
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.tight_layout()
plt.savefig(
    FIGURES / "roc_curve.png",
    dpi=150
)
plt.show()

plt.figure(figsize=(7, 5))
plt.plot(recall, precision)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.tight_layout()
plt.savefig(
    FIGURES / "precision_recall_curve.png",
    dpi=150
)
plt.show()


## Conclusion

The ROC and Precision-Recall curves provide additional evidence about model discrimination and performance across probability thresholds.


# Task 10: Calibration Curve

The calibration curve checks whether predicted probabilities are reasonably aligned with observed outcomes.


In [ ]:
prob_true, prob_pred = calibration_curve(
    y_test,
    test_prob,
    n_bins=10
)

plt.figure(figsize=(7, 5))
plt.plot(
    prob_pred,
    prob_true,
    marker="o",
    label="Model"
)
plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Perfect calibration"
)
plt.xlabel("Mean Predicted Probability")
plt.ylabel("Fraction of Positives")
plt.title("Calibration Curve")
plt.legend()
plt.tight_layout()
plt.savefig(
    FIGURES / "calibration_curve.png",
    dpi=150
)
plt.show()


## Conclusion

The calibration curve helps assess whether the model's probability estimates are useful and appropriately scaled.


# Task 11: Learning Curve

The learning curve is used to inspect how training-set size affects model performance and to identify possible bias or variance issues.


In [ ]:
train_sizes, train_scores, valid_scores = learning_curve(
    final_pipeline,
    X_train_dev,
    y_train_dev,
    cv=5,
    scoring="f1",
    train_sizes=np.linspace(0.1, 1.0, 5),
    n_jobs=-1
)

train_mean = train_scores.mean(axis=1)
valid_mean = valid_scores.mean(axis=1)

plt.figure(figsize=(8, 5))
plt.plot(
    train_sizes,
    train_mean,
    marker="o",
    label="Training F1"
)
plt.plot(
    train_sizes,
    valid_mean,
    marker="o",
    label="Validation F1"
)
plt.xlabel("Training Examples")
plt.ylabel("F1-score")
plt.title("Learning Curve")
plt.legend()
plt.tight_layout()
plt.savefig(
    FIGURES / "learning_curve.png",
    dpi=150
)
plt.show()


## Conclusion

The learning curve provides a visual check for underfitting, overfitting, and whether additional training data may improve generalization.


# Task 12: Feature Interpretation

The transformed feature names are extracted so the final model can be inspected and interpreted.


In [ ]:
feature_names = final_pipeline.named_steps[
    "preprocessor"
].get_feature_names_out()

model = final_pipeline.named_steps["model"]

if hasattr(model, "coef_"):
    importance = np.abs(
        model.coef_[0]
    )
elif hasattr(model, "feature_importances_"):
    importance = model.feature_importances_
else:
    importance = np.zeros(
        len(feature_names)
    )

feature_df = pd.DataFrame(
    {
        "Feature": feature_names,
        "Importance": importance
    }
)

feature_df = feature_df.sort_values(
    "Importance",
    ascending=False
)

feature_df.head(20).to_csv(
    RESULTS / "top_features.csv",
    index=False
)

top_features = feature_df.head(20)

plt.figure(figsize=(9, 7))
plt.barh(
    top_features["Feature"][::-1],
    top_features["Importance"][::-1]
)
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Top Feature Importance")
plt.tight_layout()
plt.savefig(
    FIGURES / "feature_importance.png",
    dpi=150
)
plt.show()

print(top_features)


## Conclusion

The most influential transformed features are identified and saved. This improves transparency and makes the final model easier to inspect.


# Task 13: Subgroup Analysis

Performance is compared across selected demographic subgroups using the untouched test predictions.


In [ ]:
subgroup_results = []

for col in ["sex", "race", "workclass"]:
    if col not in X_test.columns:
        continue

    temp = X_test.copy()

    temp["actual"] = y_test.values

    temp["predicted"] = test_pred

    for group, part in temp.groupby(col, dropna=False):
        if len(part) < 10:
            continue

        subgroup_results.append(
            {
                "Feature": col,
                "Group": str(group),
                "Rows": len(part),
                "Accuracy": accuracy_score(
                    part["actual"],
                    part["predicted"]
                ),
                "Precision": precision_score(
                    part["actual"],
                    part["predicted"],
                    zero_division=0
                ),
                "Recall": recall_score(
                    part["actual"],
                    part["predicted"],
                    zero_division=0
                ),
                "F1": f1_score(
                    part["actual"],
                    part["predicted"],
                    zero_division=0
                )
            }
        )

subgroup_df = pd.DataFrame(
    subgroup_results
)

subgroup_df.to_csv(
    RESULTS / "subgroup_analysis.csv",
    index=False
)

print(subgroup_df)


## Conclusion

Subgroup analysis provides an additional check of model behavior across important categorical groups and can highlight differences that are hidden by overall metrics.


# Task 14: Save Final Pipeline

The final trained pipeline and metadata are saved for reproducible inference.


In [ ]:
pipeline_path = MODELS / "adult_income_final_pipeline.joblib"

joblib.dump(
    final_pipeline,
    pipeline_path
)

metadata = {
    "model": best_name,
    "threshold": best_threshold,
    "random_state": RANDOM_STATE,
    "primary_metric": "F1"
}

joblib.dump(
    metadata,
    MODELS / "adult_income_metadata.joblib"
)

print("Pipeline saved to:", pipeline_path)


## Conclusion

The trained pipeline and decision threshold are saved so the same preprocessing and prediction process can be reused for future inference.


# Task 15: Production Inference Function

A reusable inference function is created. It accepts one or more unseen Adult Income records and returns predictions with probabilities.


In [ ]:
def predict_income(data):
    if isinstance(data, dict):
        data = pd.DataFrame([data])
    elif isinstance(data, pd.Series):
        data = data.to_frame().T
    else:
        data = data.copy()

    prob = final_pipeline.predict_proba(data)[:, 1]

    pred = (
        prob >= best_threshold
    ).astype(int)

    result = data.copy()

    result["probability"] = prob

    result["prediction"] = pred

    result["prediction_label"] = np.where(
        pred == 1,
        ">50K",
        "<=50K"
    )

    return result

sample_input = X_test.head(10).copy()

sample_output = predict_income(
    sample_input
)

print(sample_output)


## Conclusion

The inference function provides a simple reusable interface for predicting income on unseen records using the exact locked pipeline and threshold.


# Task 16: Leakage Verification

The final test set is checked to confirm that it was not used during model fitting or threshold selection.


In [ ]:
train_ids = set(X_train.index)
dev_ids = set(X_dev.index)
test_ids = set(X_test.index)

print(
    "Train/Test overlap:",
    len(train_ids.intersection(test_ids))
)

print(
    "Development/Test overlap:",
    len(dev_ids.intersection(test_ids))
)

print(
    "Train/Development overlap:",
    len(train_ids.intersection(dev_ids))
)

assert len(train_ids.intersection(test_ids)) == 0
assert len(dev_ids.intersection(test_ids)) == 0
assert len(train_ids.intersection(dev_ids)) == 0

print("Leakage verification passed.")


## Conclusion

The dataset splits contain no overlapping rows. The final test set remains isolated from model development, supporting a valid final evaluation.


# Final Conclusion

The Adult Income classification project is complete. The final workflow includes leakage-safe data splitting, reproducible preprocessing, model comparison, threshold selection, final test evaluation, confusion-matrix analysis, ROC and Precision-Recall curves, calibration analysis, learning curves, feature interpretation, subgroup analysis, production inference, and saved model artifacts.

The final pipeline is ready for reproducible use and submission.


# Final Submission Checklist

- Final Jupyter notebook
- `models/adult_income_final_pipeline.joblib`
- `models/adult_income_metadata.joblib`
- Final metrics table
- Confusion matrix
- Learning curve
- Calibration curve
- ROC curve
- Precision-Recall curve
- Feature interpretation visualization
- False-positive samples
- False-negative samples
- Subgroup analysis
- Working inference function
- 10 unseen examples
- Leakage verification
- Final test evaluation
